In [ ]:
import nltk

nltk.download('punkt')       # usual tokenizer
nltk.download('punkt_tab')   # fallback table NLTK wants


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr libtesseract-dev
!pip install -q sentence-transformers faiss-cpu pdfplumber pypdf2 pdf2image pytesseract nltk transformers

import re, os
from pathlib import Path
import pdfplumber
from pypdf import PdfReader
from pdf2image import convert_from_path
import pytesseract
from sentence_transformers import SentenceTransformer
import faiss
import nltk
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# NLTK fix
nltk.download('punkt')
nltk.download('punkt_tab')

PDF_PATH = "/content/Sample Lease 25-26 .pdf"
if not Path(PDF_PATH).exists(): raise FileNotFoundError(f"Cannot find {PDF_PATH}.")

def extract_visible_text(pdf_path):
    text=""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for p in pdf.pages:
                t = p.extract_text()
                if t: text+=t+"\n"
    except: pass
    if not text.strip():
        try:
            reader = PdfReader(pdf_path)
            for p in reader.pages:
                t = p.extract_text()
                if t: text+=t+"\n"
        except: pass
    return text

def extract_form_fields(pdf_path):
    fields={}
    try:
        reader = PdfReader(pdf_path)
        try:
            ff = reader.get_fields()
            if ff:
                for k,v in ff.items():
                    val = v.get('/V') if isinstance(v,dict) else v
                    if val is not None: fields[str(k).lower()]=str(val)
        except:
            if "/AcroForm" in reader.trailer["/Root"]:
                form = reader.trailer["/Root"]["/AcroForm"]
                if "/Fields" in form:
                    for f in form["/Fields"]:
                        fo = f.get_object()
                        name = fo.get("/T")
                        value = fo.get("/V")
                        if name: fields[str(name).lower()]=str(value) if value else ""
    except: pass
    return fields

def ocr_pdf_pages(pdf_path,dpi=300,max_pages=None):
    images=convert_from_path(pdf_path,dpi=dpi)
    if max_pages: images=images[:max_pages]
    page_texts,page_data=[],[]
    for img in images:
        if img.mode!="RGB": img=img.convert("RGB")
        txt=pytesseract.image_to_string(img)
        page_texts.append(txt)
        try: data=pytesseract.image_to_data(img,output_type=pytesseract.Output.DICT)
        except: data=None
        page_data.append(data)
    return page_texts,page_data

visible_text=extract_visible_text(PDF_PATH)
form_fields=extract_form_fields(PDF_PATH)
ocr_texts,ocr_page_data=ocr_pdf_pages(PDF_PATH)
ocr_combined_text="\n".join(ocr_texts)


def normalize_number_str(s):
    if not s: return None
    s=re.sub(r"[$,]","",s)
    m=re.search(r"(\d{2,6}(?:\.\d{1,2})?)",s)
    if not m: return None
    try: return float(m.group(1))
    except: return None

def plausible_currency(val,low=300,high=20000):
  return val is not None and low<=val<=high

def find_candidates_in_ocr(page_data,page_text,keywords=("rent","deposit")):
    candidates=[]
    if page_data is None:
        for ln in page_text.splitlines():
            if any(kw in ln.lower() for kw in keywords):
                for n in re.findall(r"\$?\s?(\d{2,6}(?:\.\d{1,2})?)",ln):
                    val=normalize_number_str(n)
                    candidates.append(("ocr_line",ln,val))
        return candidates
    words,page_lines=page_data,{}
    for i,txt in enumerate(words['text']):
        if not txt: continue
        key=(words.get('block_num')[i],words.get('par_num')[i],words.get('line_num')[i])
        page_lines.setdefault(key,[]).append(txt)
    for wlist in page_lines.values():
        line=" ".join(wlist)
        if any(kw in line.lower() for kw in keywords):
            for n in re.findall(r"\$?\s?(\d{2,6}(?:\.\d{1,2})?)",line):
                val=normalize_number_str(n)
                candidates.append(("ocr_line",line,val))
    return candidates

ocr_candidates=[]
for i,pdata in enumerate(ocr_page_data):
    page_txt=ocr_texts[i] if i<len(ocr_texts) else ""
    for c in find_candidates_in_ocr(pdata,page_txt,keywords=("rent","monthly rent","security deposit","deposit")):
        ocr_candidates.append((i,c[0],c[1],c[2]))

global_ocr_vals=[normalize_number_str(v) for v in re.findall(r"(?:rent|deposit).{0,40}?\$?\s?(\d{2,6}(?:\.\d{1,2})?)",ocr_combined_text,re.I) if normalize_number_str(v) is not None]


def infer_rent_and_deposit(pdf_path,text,fields,ocr_candidates,ocr_text_combined,debug=False):
    candidates={"rent":[],"deposit":[]}
    for k,v in fields.items():
        nv=normalize_number_str(v)
        if nv and plausible_currency(nv):
            if "rent" in k or "price" in k: candidates["rent"].append(("field",k,nv,100))
            if "deposit" in k or "security" in k: candidates["deposit"].append(("field",k,nv,100))
    for i,ln in enumerate([l.strip() for l in text.splitlines() if l.strip()]):
        low=ln.lower()
        for cat,patterns in [("rent",["monthly rent","rent amount","rent shall be","rent is"]),("deposit",["security deposit","deposit shall","deposit is"])]:
            if any(p in low for p in patterns) or (cat in low and "$" in low):
                for n in re.findall(r"\$?\s?(\d{2,6}(?:\.\d{1,2})?)",ln):
                    nv=normalize_number_str(n)
                    if nv and plausible_currency(nv): candidates[cat].append(("text_line",ln[:120],nv,60-i//50))
    for (pg,typ,line,val) in ocr_candidates:
        if val is None: continue
        low=str(line).lower()
        if "rent" in low or "monthly" in low: candidates["rent"].append(("ocr",f"page{pg}",val,70-pg))
        if "deposit" in low: candidates["deposit"].append(("ocr",f"page{pg}",val,70-pg))
    for cat in ["rent","deposit"]:
        lst=candidates[cat]
        if lst: lst.sort(key=lambda x:(x[3],x[2]),reverse=True)
    rent_val=candidates["rent"][0][2] if candidates["rent"] else None
    dep_val=candidates["deposit"][0][2] if candidates["deposit"] else None
    return rent_val,dep_val

rent_val,deposit_val=infer_rent_and_deposit(PDF_PATH,visible_text,form_fields,ocr_candidates,ocr_combined_text,debug=True)
print("\nInferred rent:",rent_val,"inferred deposit:",deposit_val)


embedder=SentenceTransformer("all-MiniLM-L6-v2")
def chunk_text(text,max_words=100):
    sents=sent_tokenize(text);chunks=[];cur=""
    for s in sents:
        if len((cur+" "+s).split())>max_words:
            if cur.strip(): chunks.append(cur.strip())
            cur=s
        else: cur+= " "+s
    if cur.strip(): chunks.append(cur.strip())
    return chunks

chunks=chunk_text(visible_text+"\n\n"+ocr_combined_text)
embs=embedder.encode(chunks,convert_to_numpy=True)
index=faiss.IndexFlatL2(embs.shape[1])
index.add(embs)

def retrieve_limited(query,k=4,max_tokens=400):
    top_chunks=chunks_to_use=retrieve(query,k)
    truncated=[]
    total_tokens=0
    for chunk in top_chunks:
        tokens=len(tok.tokenize(chunk))
        if total_tokens+tokens>max_tokens: break
        truncated.append(chunk)
        total_tokens+=tokens
    return truncated

def retrieve(query,k=4):
    qv=embedder.encode([query],convert_to_numpy=True)
    D,I=index.search(qv,k)
    return [chunks[idx] for idx in I[0] if 0<=idx<len(chunks)]


tok=AutoTokenizer.from_pretrained("google/flan-t5-small")
model=AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small",device_map="auto")
def run_llm(prompt,max_tokens=256):
    inputs=tok(prompt,return_tensors="pt").to(model.device)
    out=model.generate(**inputs,max_new_tokens=max_tokens,do_sample=False)
    return tok.decode(out[0],skip_special_tokens=True)


GROUND_TRUTH_PHRASES={"Parties to the Agreement":["landlord","tenant","party of the first part","party of the second part","lessor","lessee"],
"Property Description":["premises","property located at","address","unit","apartment","house","condo","suite"],
"Term / Duration":["term","lease term","commence","commencement","lease start","lease end","month-to-month","fixed term"],
"Rent":["monthly rent","rent amount","rent shall be","rent is","rent payable","monthly total rent","base rent"],
"Security Deposit":["security deposit","deposit shall","deposit is","security deposit amount"],
"Maintenance & Repairs":["maintenance","repairs","landlord shall","tenant shall"],
"Utilities Responsibility":["utilities","water","electricity","gas","sewer","trash","garbage"],
"Rules & Restrictions":["no smoking","no pets","rules and regulations","occupancy limit","restrictions"],
"Termination & Renewal":["terminate","termination","renewal","notice to vacate","notice of nonrenewal"],
"Late Fees / Penalties":["late fee","late charge","penalty","grace period"],
"Privacy & Entry Rights":["entry","landlord may enter","notice before entry","reasonable notice"],
"Insurance Requirements":["insurance","renter's insurance","liability insurance"],
"Subletting Policy":["sublet","subletting","assignment","assign"],
"Dispute Resolution":["arbitration","mediation","dispute resolution"],
"Governing Law":["governing law","laws of the state","jurisdiction","tenant protection act"],
"Signatures & Date":["signature","signed","date","tenant signature","landlord signature"],
"Parking":["parking","garage","parking space","parking permit"],
"Furnished/unfurnished":["furnished","unfurnished","furniture included"]}

def deterministic_checklist(text):
    tl=text.lower().replace("\n"," ")
    present,missing=[],[]
    for sec,phrases in GROUND_TRUTH_PHRASES.items():
        if any(ph in tl for ph in phrases): present.append(sec)
        else: missing.append(sec)
    return present,missing


def answer_question(query):
    ql=query.lower().strip()
    if any(x in ql for x in ["present","missing","what sections","sections"]):
        present,missing=deterministic_checklist(visible_text+"\n\n"+ocr_combined_text)
        return f"Present: {'; '.join(present) if present else 'None'}\nMissing: {'; '.join(missing) if missing else 'None'}"
    if "rent" in ql or "deposit" in ql:
        r,d=infer_rent_and_deposit(PDF_PATH,visible_text,form_fields,ocr_candidates,ocr_combined_text,debug=False)
        if "rent" in ql: return f"Rent: ${r:.2f}" if r else "Rent: Not found"
        if "deposit" in ql: return f"Security deposit: ${d:.2f}" if d else "Security deposit: Not found"
    top=retrieve_limited(query,k=4,max_tokens=400)
    context="\n\n---\n\n".join(top)
    prompt=f"You are a strict document assistant. ANSWER ONLY using the CONTEXT below. If the answer cannot be found, reply exactly: Not mentioned in the document.\n\nCONTEXT:\n{context}\n\nQUESTION:\n{query}\n\nANSWER (concise):"
    resp=run_llm(prompt,max_tokens=256).strip()
    if not resp or "not mentioned" in resp.lower(): return "Not mentioned in the document."
    return resp.splitlines()[0].strip()

print("\nSystem ready. Ask questions (type 'exit'):\n")
while True:
    q=input("Your question (or 'exit'): ").strip()
    if not q: continue
    if q.lower() in ("exit","quit"): break
    print("\nAnswer:\n",answer_question(q))
    print("\n"+"-"*60+"\n")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Inferred rent: 3100.0 inferred deposit: 3100.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



System ready. Ask questions (type 'exit'):

Your question (or 'exit'): What sections are present

Answer:
 Present: Parties to the Agreement; Property Description; Term / Duration; Rent; Security Deposit; Maintenance & Repairs; Utilities Responsibility; Rules & Restrictions; Termination & Renewal; Late Fees / Penalties; Privacy & Entry Rights; Insurance Requirements; Subletting Policy; Signatures & Date; Parking; Furnished/unfurnished
Missing: Dispute Resolution; Governing Law

------------------------------------------------------------

Your question (or 'exit'): What sections are missing?

Answer:
 Present: Parties to the Agreement; Property Description; Term / Duration; Rent; Security Deposit; Maintenance & Repairs; Utilities Responsibility; Rules & Restrictions; Termination & Renewal; Late Fees / Penalties; Privacy & Entry Rights; Insurance Requirements; Subletting Policy; Signatures & Date; Parking; Furnished/unfurnished
Missing: Dispute Resolution; Governing Law

--------------

KeyboardInterrupt: Interrupted by user

In [ ]:
!pip install ollama